In [ ]:
library(tidyverse)
library(mvtnorm)
library(ggrepel)
library(readxl)

In [ ]:
rr <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL.xlsx", sheet = "Table5") 
tet <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL.xlsx", sheet = "Table6") 


In [ ]:
# ============================================================
# Supplementary figure (ceiling-only version)
# Observed RR vs prevalence, with the 1/K ceiling as the only curve.
# Point labels carry the tetrachoric r; the ceiling shows that RR is
# bounded by prevalence, so a large RR at low K is near its ceiling.
# ============================================================
 
library(mvtnorm)
library(dplyr)
library(tidyr)
library(stringr)
library(purrr)
library(ggplot2)
library(ggrepel)
 
# ---- liability-model helpers ----
upper_orthant <- function(T, r)
  as.numeric(pmvnorm(lower = c(T, T), upper = c(Inf, Inf),
                     corr = matrix(c(1, r, r, 1), 2)))
 
lambda_pred <- function(K, r) { T <- qnorm(1 - K); upper_orthant(T, r) / K^2 }
 
# Optional consistency check: r implied by plotted (K, RR) vs labelled tet_r.
# Kept as a data sanity print even though no iso-r curves are drawn.
r_from_RR <- function(K, RR) {
  if (is.na(K) || is.na(RR) || K <= 0 || K >= 1) return(NA_real_)
  if (RR <= 1) return(0)
  out <- tryCatch(uniroot(function(r) lambda_pred(K, r) - RR,
                          lower = 0, upper = 0.999, tol = 1e-6),
                  error = function(e) NULL)
  if (is.null(out)) NA_real_ else out$root
}
 
# ---- merge the two analyses ----
rr2 <- rr %>% transmute(Outcome = str_replace_all(Outcome, "_", " "),
                        relationship_type, K = risk_in_sample,
                        RR = recurrence_risk_ratio, RR_lo = ci_lower,
                        RR_hi = ci_upper, p_rr = p_value)
tet2 <- tet %>% transmute(Outcome = str_replace_all(outcome, "_", " "),
                          relationship_type, tet_r = tetrachoric_r, p_tet = p)
m <- inner_join(rr2, tet2, by = c("Outcome", "relationship_type")) %>%
  mutate(sig_tet = p_tet < 0.05)
 
# ---- (optional) print consistency check ----
m %>%
  mutate(r_implied = map2_dbl(K, RR, r_from_RR), gap = tet_r - r_implied) %>%
  transmute(Outcome, relationship_type, K = round(K, 3), RR = round(RR, 2),
            tet_r = round(tet_r, 2), r_implied = round(r_implied, 2),
            gap = round(gap, 2)) %>%
  arrange(desc(abs(gap))) %>%
  print(n = Inf)
 
# ---- aesthetics ----
pheno_lv  <- c("Discontinuation","Switching","Continuation",
               "TCA Continuation","SSRI Continuation","SNRI Continuation")
pheno_col <- c("Discontinuation"="#7E8AA2","Switching"="#E07A5F","Continuation"="#3D8C7E",
               "TCA Continuation"="#C9A227","SSRI Continuation"="#2E6F95","SNRI Continuation"="#C44E52")
rel_lab   <- c(Any="All", full_sibling="Full sibling",
               parent_child="Parent-child", grandparent_grandchild="Grandparent-grandchild")
rel_shape <- c("All"=21,"Full sibling"=22,"Parent-child"=24,"Grandparent-grandchild"=23)
off       <- c(Any=-0.013, full_sibling=-0.0043, parent_child=0.0043, grandparent_grandchild=0.013)
 
m <- m %>%
  mutate(Outcome = factor(Outcome, levels = pheno_lv),
         rel     = factor(recode(relationship_type, !!!rel_lab),
                          levels = c("All","Full sibling","Parent-child","Grandparent-grandchild")),
         xoff    = K + recode(relationship_type, !!!off),
         rlab    = sprintf("r=%.2f", tet_r),
         fill_col = ifelse(sig_tet, pheno_col[as.character(Outcome)], "white"))
 
# ---- ceiling only ----
y_top <- 5.6
ceil  <- tibble(K = seq(0.03, 0.74, length.out = 300)) %>% mutate(RR = 1/K)
ceil_lab <- tibble(K = 0.50, RR = 1/0.50,
                   label = "ceiling: RR = 1/K\n(perfect concordance, r = 1)")
 
p <- ggplot() +
  geom_line(data = ceil, aes(K, RR), linetype = "dotted",
            colour = "grey35", linewidth = 0.5) +
  geom_text(data = ceil_lab, aes(K, RR, label = label),
            colour = "grey35", size = 3, hjust = 0, vjust = -0.2,
            lineheight = 0.9) +
  geom_hline(yintercept = 1, linetype = "dashed", colour = "black",
             alpha = 0.4, linewidth = 0.3) +
  geom_errorbar(data = m, aes(x = xoff, ymin = RR_lo, ymax = RR_hi, colour = Outcome),
                width = 0, linewidth = 0.5, alpha = 0.8) +
  geom_point(data = m, aes(x = xoff, y = RR, colour = Outcome, shape = rel),
             fill = m$fill_col, size = 2.8, stroke = 0.9) +
  geom_text_repel(data = m, aes(x = xoff, y = RR, label = rlab, colour = Outcome),
                  size = 3, show.legend = FALSE, min.segment.length = 0,
                  segment.size = 0.2, box.padding = 0.3, max.overlaps = Inf) +
  geom_point(data = m, aes(x = xoff, y = RR, alpha = sig_tet),
             shape = 21, colour = NA, fill = NA, size = 2.8) +
  scale_colour_manual(values = pheno_col, name = "Phenotype") +
  scale_shape_manual(values = rel_shape, name = "Relationship") +
  scale_alpha_manual(values = c(`TRUE` = 0, `FALSE` = 0),
                     breaks = c(TRUE, FALSE),
                     labels = c("p < 0.05", "n.s."),
                     name = "Tetrachoric") +
  guides(
    colour = guide_legend(order = 1, override.aes = list(shape = 21, fill = pheno_col)),
    shape  = guide_legend(order = 2, override.aes = list(fill = "grey40")),
    alpha  = guide_legend(order = 3, override.aes =
               list(shape = 21, colour = "grey25", fill = c("grey25", "white"), alpha = 1))
  ) +
  coord_cartesian(xlim = c(0.02, 0.78), ylim = c(0.5, y_top)) +
  labs(x = "Phenotype prevalence in sample, K",
       y = "Recurrence risk ratio, RR = K1 / K",
       title = "Recurrence-risk ratio is bounded by prevalence",
       subtitle = "Dotted line is the RR ceiling (1/K); tetrachoric r is labelled at each point") +
  theme_classic(base_size = 11) +
  theme(plot.title    = element_text(face = "bold", size = 13),
        plot.subtitle = element_text(size = 9, colour = "grey30"))
 
p
 
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_LL_tetr_and_RR.png",
       plot = p, width = 9, height = 6, device = "png", units = "in")

In [ ]:
# =====================================================================
# Supplementary figures for the familial-aggregation analysis (Lifelines)
#   S1: tetrachoric correlations (forest plot)  -- from `tet`
#   S2: recurrence-risk ratios (K vs K1 bars)   -- from `rr`
# =====================================================================
 
# common ordering / labels -------------------------------------------------
outcome_lv <- c("Continuation", "Discontinuation", "Switching",
                "SSRI Continuation", "TCA Continuation", "SNRI Continuation")
rel_lab    <- c(Any = "All", full_sibling = "Full sibling",
                parent_child = "Parent-child",
                grandparent_grandchild = "Grandparent-grandchild")
rel_col    <- c("All" = "#1B9E8F", "Full sibling" = "#2E6F95",
                "Parent-child" = "#C44E72", "Grandparent-grandchild" = "#E07A2F")
rel_col <- c(
  "All"                    = "#5B3A8E",  # indigo-violet
  "Grandparent-grandchild" = "#B5398E",  # magenta
  "Full sibling"           = "#6E9B34",  # olive / lime-green
  "Parent-child"           = "#9A6324"   # brown
)
 
# =====================================================================
# S1 — TETRACHORIC CORRELATION FOREST PLOT  (from `tet`)
#   tet cols: outcome, relationship_type, tetrachoric_r, ci_lower, ci_upper, p
# =====================================================================
tet
tet_p <- tet %>%
  transmute(Outcome = str_replace_all(outcome, "_", " "),
            rel = recode(relationship_type, !!!rel_lab),
            r = tetrachoric_r, lo = ci_lower, hi = ci_upper, p = p) %>%
  mutate(sig = (lo > 0 | hi < 0),
         Outcome = factor(Outcome, levels = outcome_lv),
         rel = factor(rel, levels = c("Parent-child","Full sibling",
                                      "Grandparent-grandchild","All")),
         # filled if significant, hollow otherwise (shapes 21-24 take a fill)
         fill_col = ifelse(sig, rel_col[as.character(rel)], "white"))
tet_p
pS1 <- ggplot(tet_p, aes(x = r, y = rel, colour = rel)) +
  geom_vline(xintercept = 0, linetype = "dashed", colour = "black") +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.25, linewidth = 0.8) +
  geom_point(aes(shape = rel), fill = tet_p$fill_col, size = 3, stroke = 1.1) +
  scale_shape_manual(values = c("Parent-child" = 21, "Full sibling" = 21,
                                "Grandparent-grandchild" = 21, "All" = 21),
                     guide = "none") +
  scale_colour_manual(values = rel_col, guide = "none") +
  facet_wrap(~ Outcome, nrow = 2) +
  labs(x = "Tetrachoric correlation (95% CI)", y = NULL,
       title = "Familial aggregation of antidepressant outcomes") +
  theme_classic() +
theme(strip.text = element_text(face = "bold"),
      strip.background = element_rect(colour = "black", linewidth = 0.4),
      axis.text.y  = element_text(colour = "black", size = 12),  # tick numbers
      axis.title = element_text(size = 14),                    # axis label text
      plot.title = element_text(face = "bold"))
pS1
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_LL_tetr.png",
       plot   = pS1,
       width  = 8,
       height = 5,
       device = "png",
       units  = "in")
 
# =====================================================================
# S2 — RECURRENCE-RISK RATIO BAR CHART  (from `rr`)
#   rr cols: Outcome, relationship_type, risk_in_sample (K),
#            risk_in_relatives (K1), recurrence_risk_ratio (RR),
#            ci_lower, ci_upper
# =====================================================================
rel_lv <- c("All", "Full sibling", "Parent-child", "Grandparent-grandchild")
 
# long form: one bar for K (sample) and one for K1 (relatives)
rr_long <- rr %>%
  transmute(Outcome = str_replace_all(Outcome, "_", " "),
            rel = recode(relationship_type, !!!rel_lab),
            K = risk_in_sample, K1 = risk_in_relatives) %>%
  pivot_longer(c(K, K1), names_to = "sample", values_to = "risk") %>%
  mutate(sample = recode(sample, K = "K (sample)", K1 = "K1 (relatives)"))
 
# RR labels, positioned above the taller of the two bars
rr_lab <- rr %>%
  transmute(Outcome = str_replace_all(Outcome, "_", " "),
            rel = recode(relationship_type, !!!rel_lab),
            RR = recurrence_risk_ratio, lo = ci_lower, hi = ci_upper,
            y = pmax(risk_in_sample, risk_in_relatives) + 0.02) %>%
  mutate(label = sprintf("RR=%.2f\n(%.2f-%.2f)", RR, lo, hi))
 
rr_long <- rr_long %>%
  mutate(Outcome = factor(Outcome, levels = outcome_lv),
         rel = factor(rel, levels = rel_lv))
rr_lab  <- rr_lab %>%
  mutate(Outcome = factor(Outcome, levels = outcome_lv),
         rel = factor(rel, levels = rel_lv))
 
pS2 <- ggplot(rr_long, aes(x = rel, y = risk, fill = sample)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7, alpha = 0.85) +
  geom_text(data = rr_lab, aes(x = rel, y = y, label = label),
            inherit.aes = FALSE, size = 2.6, vjust = -0.2, lineheight = 0.85) +
  scale_fill_manual(values = c("K (sample)" = "grey65",
                               "K1 (relatives)" = "#2A2A8A"), name = NULL) +
  facet_wrap(~ Outcome, nrow = 2, scales = "free_x") +
  coord_cartesian(ylim = c(0, 0.85)) +
  labs(x = "Relationship", y = "Lifetime risk (proportion)",
       title = "Recurrence risk ratios in Lifelines") +
  theme_classic(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, colour = "black"),
        axis.text.y = element_text(colour = "black"),
        axis.title = element_text(face = "bold"),
        strip.text = element_text(face = "bold"),
        strip.background = element_rect(colour = "black", linewidth = 0.4),
        legend.position = "right")
pS2
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_LL_RR.png",
       plot   = pS2,
       width  = 10,
       height = 7,
       device = "png",
       units  = "in")